# TPoliCynav: The TPC Policy Navigator
## Checkpoint 1: Foundation & Data Pipeline (Prelim Performance Task)

**Course:** IS Professional Elective #4 (Generative AI)  
**Team:** TEAM KUYS  
**Members:** Eduardo Avenido Jr., Christian Paul Larosa, Nielbert Garcia, Marcel Avergonzado  

---
### Overview
This notebook demonstrates the complete end-to-end data pipeline for Checkpoint 1:
1. **Data Ingestion & PDF Extraction**
2. **Text Cleaning & Noise Removal** (with before/after examples)
3. **Text Normalization** (with before/after examples)
4. **Sentence-Level Tokenization**
5. **Contextual Text Chunking with Overlap**
6. **Dense Embedding Generation using BAAI/bge-small-en-v1.5**
7. **Semantic Similarity & Retrieval Demonstration**

In [ ]:
# Step 0: Imports and Environment Setup
import os
import re
import json
import unicodedata
import numpy as np
from sentence_transformers import SentenceTransformer

RAW_DATA_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)
print('Environment initialized.')

## 1. Inspecting Raw Data Corpus (30+ Pages)
Let's list all raw files collected from official institutional and municipal sources.

In [ ]:
raw_files = [f for f in os.listdir(RAW_DATA_DIR) if f.endswith('.txt')]
print(f'Found {len(raw_files)} raw documents:\n')
for f in raw_files:
    size = os.path.getsize(os.path.join(RAW_DATA_DIR, f))
    print(f' - {f:<30} ({size:,} bytes)')

## 2. Text Preprocessing: Noise Removal
We strip out PDF page headers, footers, form feed markers, and broken hyphenation.

In [ ]:
def remove_noise(text: str) -> str:
    text = re.sub(r'---\s*Page\s*\d+\s*---', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\x0c', '', text)
    text = re.sub(r'[\x00-\x08\x0b\x0e-\x1f]', '', text)
    text = re.sub(r'[-_]{5,}', '', text)
    text = re.sub(r'(\w)-\s*\n\s*(\w)', r'\1\2', text)
    return text.strip()

# Demonstration on sample text
sample_raw = '''--- Page 14 ---\nTPC STUDENT HANDBOOK\nDis-\nciplinary matrix for stu-\ndents.\n\n\n\n42\n--- Page 15 ---'''
print('🔴 BEFORE Noise Removal:\n', repr(sample_raw))
print('\n🟢 AFTER Noise Removal:\n', repr(remove_noise(sample_raw)))

## 3. Text Normalization
Standardizing Unicode characters, smart quotes, dashes, bullet points, and whitespace.

In [ ]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    text = re.sub(r'[•●○▪▸►◦]', '-', text)
    text = re.sub(r'[^\S\n]+', ' ', text)
    lines = [line.strip() for line in text.split('\n')]
    return '\n'.join(lines).strip()

sample_dirty = '• Section 1: “Students’ Rights” – Mandatory Attendance   '
print('🔴 BEFORE Normalization:\n', repr(sample_dirty))
print('\n🟢 AFTER Normalization:\n', repr(normalize_text(sample_dirty)))

## 4. Sentence Tokenization & Chunking
We segment text into sentences while protecting abbreviations (e.g., GWA., No., Sec.), then group sentences into ~400-word chunks with 80-word overlap.

In [ ]:
def tokenize_sentences(text: str) -> list:
    abbreviations = [r'No\.', r'Sec\.', r'Art\.', r'GWA\.', r'AY\.']
    for i, abbr in enumerate(abbreviations):
        text = re.sub(abbr, f'ABBR{i}HOLD', text, flags=re.IGNORECASE)
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    restored = []
    for sent in sentences:
        for i, abbr in enumerate(abbreviations):
            sent = sent.replace(f'ABBR{i}HOLD', abbr.replace('\\', ''))
        if sent.strip():
            restored.append(sent.strip())
    return restored

sample_doc = 'The student must maintain a 90% GWA. Applications close on Jan. 15. Late filings will not be accepted.'
sents = tokenize_sentences(sample_doc)
print(f'Tokenized into {len(sents)} sentences:')
for i, s in enumerate(sents, 1):
    print(f' [{i}] {s}')

## 5. Embedding Generation with BAAI/bge-small-en-v1.5
We convert the processed policy chunks into 384-dimensional dense vectors.

### Why BAAI/bge-small-en-v1.5?
- **High Retrieval Accuracy:** Evaluated at the top tier of MTEB (Massive Text Embedding Benchmark) for sentence transformers under 250MB.
- **Fast Inference & Lightweight:** 384 dimensions allow low-latency local encoding and efficient cloud indexing.
- **Commercial Open Source:** Permissive MIT license suitable for real-world deployment.

In [ ]:
print('Loading BAAI/bge-small-en-v1.5...')
model = SentenceTransformer('BAAI/bge-small-en-v1.5')

# Load our actual preprocessed chunks
with open('../data/processed/chunks.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f'Loaded {len(chunks)} real policy chunks.')

# Load saved embeddings matrix
embeddings = np.load('../data/processed/embeddings.npy')
print(f'Embeddings Shape: {embeddings.shape}')
print(f'Vector Dimension: {embeddings.shape[1]}')

## 6. Semantic Similarity & Retrieval Demonstration
Testing natural language retrieval across semantic gaps (e.g. searching 'torn pants' finds 'ripped jeans').

In [ ]:
query = 'What is the dress code for the entrance exam?'
query_vec = model.encode(query, normalize_embeddings=True)

similarities = np.dot(embeddings, query_vec)
top_indices = np.argsort(similarities)[::-1][:3]

print(f'Query: "{query}"\n')
for rank, idx in enumerate(top_indices, 1):
    chunk = chunks[idx]
    print(f'🏆 Rank {rank} (Score: {similarities[idx]:.4f})')
    print(f'📄 Source: {chunk["source"]}')
    print(f'📝 Text: {chunk["text"][:200]}...\n')